In [0]:
warehouses = [
    "Berlin", "Hamburg", "Munich", "Cologne", "Frankfurt",
    "Stuttgart", "Düsseldorf", "Leipzig", "Dortmund", "Essen",
    "Bremen", "Dresden", "Hannover", "Nuremberg", "Duisburg",
    "Bochum", "Wuppertal", "Bielefeld", "Bonn", "Münster",
    "Karlsruhe", "Mannheim", "Augsburg", "Wiesbaden", "Gelsenkirchen",
    "Mönchengladbach", "Braunschweig", "Chemnitz", "Kiel", "Aachen"
]

: 

In [0]:
# assign capacities
import numpy as np
import pandas as pd

np.random.seed(42)

capacity_data = []

for city in warehouses:
    if city in ["Berlin", "Hamburg", "Munich", "Frankfurt"]:
        capacity = np.random.randint(110000, 140000)
    elif city in ["Cologne", "Stuttgart", "Düsseldorf"]:
        capacity = np.random.randint(90000, 110000)
    else:
        capacity = np.random.randint(70000, 95000)
    
    capacity_data.append({
        "warehouse": city,
        "capacity": capacity
    })

capacity_df = pd.DataFrame(capacity_data)

In [0]:
# generate forecast data
weeks = list(range(23, 33))  # 10 weeks

rows = []

for _, row in capacity_df.iterrows():
    city = row["warehouse"]
    
    base = np.random.randint(60000, 100000)
    
    for week in weeks:
        seasonality = 15000 * np.sin(2 * np.pi * week / 52)
        noise = np.random.normal(0, 8000)
        
        forecast = base + seasonality + noise
        
        rows.append({
            "week": week,
            "warehouse": city,
            "forecast": int(max(20000, forecast))
        })

df = pd.DataFrame(rows)

In [0]:
# scenarios
# overflow
overflow_cities = ["Berlin", "Leipzig", "Dresden"]
for city in overflow_cities:
    df.loc[(df["warehouse"] == city) & (df["week"] == 30), "forecast"] *= 1.5

# underflow
underutilized = ["Bremen", "Kiel", "Augsburg"]
for city in underutilized:
    df.loc[(df["warehouse"] == city) & (df["week"] == 30), "forecast"] *= 0.6

# big spike
df.loc[(df["warehouse"] == "Berlin") & (df["week"] == 29), "forecast"] *= 1.4

# near capacity
near_capacity = ["Frankfurt", "Cologne"]
for city in near_capacity:
    df.loc[(df["warehouse"] == city) & (df["week"] == 30), "forecast"] *= 1.1

/home/spark-74414b57-3928-45bf-aa1d-cd/.ipykernel/7266/command-4921369846161599-1098705546:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[73468.5]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[(df["warehouse"] == city) & (df["week"] == 30), "forecast"] *= 1.5


In [0]:
# add features
df = df.merge(capacity_df, on="warehouse")

df = df.sort_values(["warehouse", "week"])

df["prev_forecast"] = df.groupby("warehouse")["forecast"].shift(1)

df["change_pct"] = (
    (df["forecast"] - df["prev_forecast"]) /
    df["prev_forecast"]
)

df["utilization"] = df["forecast"] / df["capacity"]

In [0]:
# add status labels
def classify(util):
    if util > 1.1:
        return "critical_overflow"
    elif util > 1.0:
        return "overflow"
    elif util > 0.9:
        return "high"
    else:
        return "normal"

df["status"] = df["utilization"].apply(classify)

In [0]:
df

,week,warehouse,forecast,capacity,prev_forecast,change_pct,utilization,status
290,23,Aachen,60892.0,94233,NaN,NaN,0.646186,normal
291,24,Aachen,64686.0,94233,60892.0,0.062307,0.686447,normal
292,25,Aachen,46254.0,94233,64686.0,-0.284946,0.490847,normal
293,26,Aachen,75330.0,94233,46254.0,0.628616,0.799401,normal
294,27,Aachen,56886.0,94233,75330.0,-0.244843,0.603674,normal
...,...,...,...,...,...,...,...,...
165,28,Wuppertal,95102.0,71685,98317.0,-0.032700,1.326665,critical_overflow
166,29,Wuppertal,93477.0,71685,95102.0,-0.017087,1.303997,critical_overflow
167,30,Wuppertal,98741.0,71685,93477.0,0.056313,1.377429,critical_overflow
168,31,Wuppertal,87649.0,71685,98741.0,-0.112334,1.222697,critical_overflow


# Knowledge base

In [0]:
# add documents
def generate_warehouse_doc(row):
    
    status_text = {
        "critical_overflow": "This warehouse is significantly over capacity.",
        "overflow": "This warehouse exceeds its capacity.",
        "high": "This warehouse is operating close to capacity.",
        "normal": "This warehouse is operating within capacity limits."
    }
    
    change_text = ""
    if pd.notna(row["change_pct"]):
        if row["change_pct"] > 0.2:
            change_text = "There is a strong increase in inbound compared to last week."
        elif row["change_pct"] < -0.2:
            change_text = "There is a significant decrease in inbound compared to last week."
        else:
            change_text = "Inbound volume is relatively stable compared to last week."
    
    return f"""
        Week {int(row['week'])} - Warehouse {row['warehouse']}

        Forecasted inbound volume: {row['forecast']:,} units.
        Capacity: {row['capacity']:,} units.
        Utilization: {row['utilization']:.0%}.

        {status_text[row['status']]}

        {change_text}
        """
df["document"] = df.apply(generate_warehouse_doc, axis=1)

In [0]:
# add network summary
def generate_network_summary(df_week):
    
    week = int(df_week["week"].iloc[0])
    
    over_capacity = df_week[df_week["utilization"] > 1]["warehouse"].tolist()
    high = df_week[(df_week["utilization"] > 0.9) & (df_week["utilization"] <= 1)]["warehouse"].tolist()
    underutilized = df_week[df_week["utilization"] < 0.7]["warehouse"].tolist()
    
    return f"""
        Week {week} - Network Summary

        Warehouses over capacity:
        {", ".join(over_capacity) if over_capacity else "None"}

        Warehouses near capacity:
        {", ".join(high) if high else "None"}

        Underutilized warehouses:
        {", ".join(underutilized) if underutilized else "None"}
        """

network_docs = []

for week, group in df.groupby("week"):
    doc = generate_network_summary(group)
    network_docs.append({
        "text": doc,
        "week": week,
        "doc_type": "network_summary"
    })

In [0]:
display(network_docs)

doc_type,text,week
network_summary,"Week 23 - Network Summary Warehouses over capacity: Augsburg, Bielefeld, Bochum, Bonn, Bremen, Dortmund, Karlsruhe, Mannheim, Mönchengladbach, Münster, Wuppertal Warehouses near capacity: Duisburg, Essen, Gelsenkirchen, Nuremberg Underutilized warehouses: Aachen, Berlin, Cologne, Frankfurt, Hamburg",23
network_summary,"Week 24 - Network Summary Warehouses over capacity: Augsburg, Bielefeld, Bonn, Braunschweig, Bremen, Dortmund, Duisburg, Gelsenkirchen, Karlsruhe, Mannheim, Münster, Wuppertal Warehouses near capacity: Chemnitz, Dresden, Essen, Hannover, Nuremberg Underutilized warehouses: Aachen, Berlin, Cologne, Frankfurt, Hamburg, Munich",24
network_summary,"Week 25 - Network Summary Warehouses over capacity: Augsburg, Bielefeld, Bonn, Bremen, Chemnitz, Dortmund, Gelsenkirchen, Karlsruhe, Mannheim, Münster, Nuremberg, Wuppertal Warehouses near capacity: Bochum, Dresden, Duisburg, Düsseldorf, Essen, Hannover, Leipzig, Mönchengladbach Underutilized warehouses: Aachen, Berlin, Cologne, Frankfurt, Hamburg, Kiel, Munich, Wiesbaden",25
network_summary,"Week 26 - Network Summary Warehouses over capacity: Augsburg, Bielefeld, Bremen, Dortmund, Karlsruhe, Mannheim, Mönchengladbach, Münster, Wuppertal Warehouses near capacity: Bonn, Gelsenkirchen, Nuremberg Underutilized warehouses: Berlin, Bochum, Cologne, Frankfurt, Hamburg, Munich, Stuttgart, Wiesbaden",26
network_summary,"Week 27 - Network Summary Warehouses over capacity: Augsburg, Bielefeld, Bremen, Dortmund, Essen, Karlsruhe, Mannheim, Münster, Wuppertal Warehouses near capacity: Gelsenkirchen, Hannover, Leipzig, Mönchengladbach, Nuremberg, Stuttgart Underutilized warehouses: Aachen, Berlin, Braunschweig, Frankfurt, Hamburg, Kiel, Munich, Wiesbaden",27
network_summary,"Week 28 - Network Summary Warehouses over capacity: Augsburg, Bielefeld, Bonn, Bremen, Dortmund, Hannover, Karlsruhe, Mannheim, Münster, Wuppertal Warehouses near capacity: Braunschweig, Essen, Nuremberg Underutilized warehouses: Aachen, Berlin, Cologne, Duisburg, Frankfurt, Hamburg, Kiel, Munich, Stuttgart",28
network_summary,"Week 29 - Network Summary Warehouses over capacity: Augsburg, Bielefeld, Bremen, Dortmund, Karlsruhe, Mannheim, Münster, Wuppertal Warehouses near capacity: Braunschweig, Gelsenkirchen Underutilized warehouses: Aachen, Essen, Frankfurt, Hamburg, Kiel, Munich, Nuremberg, Stuttgart",29
network_summary,"Week 30 - Network Summary Warehouses over capacity: Bielefeld, Dortmund, Leipzig, Mannheim, Wuppertal Warehouses near capacity: Braunschweig, Essen, Münster, Nuremberg Underutilized warehouses: Aachen, Berlin, Frankfurt, Hamburg, Kiel, Munich, Stuttgart, Wiesbaden",30
network_summary,"Week 31 - Network Summary Warehouses over capacity: Bielefeld, Bremen, Essen, Karlsruhe, Mannheim, Wuppertal Warehouses near capacity: Münster Underutilized warehouses: Aachen, Berlin, Braunschweig, Chemnitz, Cologne, Düsseldorf, Frankfurt, Hamburg, Hannover, Kiel, Leipzig, Munich, Stuttgart, Wiesbaden",31
network_summary,"Week 32 - Network Summary Warehouses over capacity: Bielefeld, Dortmund, Karlsruhe, Leipzig, Mannheim, Wuppertal Warehouses near capacity: Augsburg, Bonn, Bremen, Essen, Gelsenkirchen, Mönchengladbach, Münster Underutilized warehouses: Aachen, Berlin, Bochum, Cologne, Frankfurt, Hamburg, Kiel, Munich, Nuremberg, Stuttgart, Wiesbaden",32


In [0]:
# combine
knowledge_base = []

# warehouse docs
for _, row in df.iterrows():
    knowledge_base.append({
        "text": row["document"],
        "warehouse": row["warehouse"],
        "week": int(row["week"]),
        "doc_type": "warehouse"
    })

# network docs
knowledge_base.extend(network_docs)

In [0]:
for doc in knowledge_base[:3]:
    print(doc["text"])


        Week 23 - Warehouse Aachen

        Forecasted inbound volume: 60,892.0 units.
        Capacity: 94,233 units.
        Utilization: 65%.

        This warehouse is operating within capacity limits.

        
        

        Week 24 - Warehouse Aachen

        Forecasted inbound volume: 64,686.0 units.
        Capacity: 94,233 units.
        Utilization: 69%.

        This warehouse is operating within capacity limits.

        Inbound volume is relatively stable compared to last week.
        

        Week 25 - Warehouse Aachen

        Forecasted inbound volume: 46,254.0 units.
        Capacity: 94,233 units.
        Utilization: 49%.

        This warehouse is operating within capacity limits.

        There is a significant decrease in inbound compared to last week.
        


In [0]:
display(knowledge_base)

doc_type,text,warehouse,week
warehouse,"Week 23 - Warehouse Aachen Forecasted inbound volume: 60,892.0 units. Capacity: 94,233 units. Utilization: 65%. This warehouse is operating within capacity limits.",Aachen,23
warehouse,"Week 24 - Warehouse Aachen Forecasted inbound volume: 64,686.0 units. Capacity: 94,233 units. Utilization: 69%. This warehouse is operating within capacity limits. Inbound volume is relatively stable compared to last week.",Aachen,24
warehouse,"Week 25 - Warehouse Aachen Forecasted inbound volume: 46,254.0 units. Capacity: 94,233 units. Utilization: 49%. This warehouse is operating within capacity limits. There is a significant decrease in inbound compared to last week.",Aachen,25
warehouse,"Week 26 - Warehouse Aachen Forecasted inbound volume: 75,330.0 units. Capacity: 94,233 units. Utilization: 80%. This warehouse is operating within capacity limits. There is a strong increase in inbound compared to last week.",Aachen,26
warehouse,"Week 27 - Warehouse Aachen Forecasted inbound volume: 56,886.0 units. Capacity: 94,233 units. Utilization: 60%. This warehouse is operating within capacity limits. There is a significant decrease in inbound compared to last week.",Aachen,27
warehouse,"Week 28 - Warehouse Aachen Forecasted inbound volume: 64,007.0 units. Capacity: 94,233 units. Utilization: 68%. This warehouse is operating within capacity limits. Inbound volume is relatively stable compared to last week.",Aachen,28
warehouse,"Week 29 - Warehouse Aachen Forecasted inbound volume: 60,758.0 units. Capacity: 94,233 units. Utilization: 64%. This warehouse is operating within capacity limits. Inbound volume is relatively stable compared to last week.",Aachen,29
warehouse,"Week 30 - Warehouse Aachen Forecasted inbound volume: 65,059.0 units. Capacity: 94,233 units. Utilization: 69%. This warehouse is operating within capacity limits. Inbound volume is relatively stable compared to last week.",Aachen,30
warehouse,"Week 31 - Warehouse Aachen Forecasted inbound volume: 49,179.0 units. Capacity: 94,233 units. Utilization: 52%. This warehouse is operating within capacity limits. There is a significant decrease in inbound compared to last week.",Aachen,31
warehouse,"Week 32 - Warehouse Aachen Forecasted inbound volume: 52,850.0 units. Capacity: 94,233 units. Utilization: 56%. This warehouse is operating within capacity limits. Inbound volume is relatively stable compared to last week.",Aachen,32


In [0]:
# simple prompt
def get_context(kb, week):
    return [doc["text"] for doc in kb if doc["week"] == week]

context = "\n\n".join(get_context(knowledge_base, 30))

prompt = f"""
    Answer the question using the context below.

    Context:
    {context}

    Question:
    Which warehouses exceed capacity?
    """

In [0]:
display(prompt)

'\n    Answer the question using the context below.\n\n    Context:\n    \n        Week 30 - Warehouse Aachen\n\n        Forecasted inbound volume: 65,059.0 units.\n        Capacity: 94,233 units.\n        Utilization: 69%.\n\n        This warehouse is operating within capacity limits.\n\n        Inbound volume is relatively stable compared to last week.\n        \n\n\n        Week 30 - Warehouse Augsburg\n\n        Forecasted inbound volume: 54,355.799999999996 units.\n        Capacity: 76,420 units.\n        Utilization: 71%.\n\n        This warehouse is operating within capacity limits.\n\n        There is a significant decrease in inbound compared to last week.\n        \n\n\n        Week 30 - Warehouse Berlin\n\n        Forecasted inbound volume: 72,347.0 units.\n        Capacity: 133,654 units.\n        Utilization: 54%.\n\n        This warehouse is operating within capacity limits.\n\n        There is a significant decrease in inbound compared to last week.\n        \n\n\n      

# Store data

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS llmops_forecasting;

CREATE SCHEMA IF NOT EXISTS llmops_forecasting.inbound_planning;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS llmops_forecasting.inbound_planning.forecast_data (
    week INT,
    warehouse STRING,
    forecast INT,
    capacity INT,
    prev_forecast INT,
    change_pct DOUBLE,
    utilization DOUBLE,
    status STRING
)
USING DELTA;

In [0]:
%sql
DROP TABLE IF EXISTS llmops_forecasting.inbound_planning.knowledge_base;
CREATE TABLE IF NOT EXISTS llmops_forecasting.inbound_planning.knowledge_base (
    id STRING,
    week INT,
    warehouse STRING,
    doc_type STRING,
    text STRING
)
USING DELTA;

In [0]:
from pyspark.sql import functions as F

spark_df = (
    spark.createDataFrame(df)
    .select(
        F.col("week").cast("int").alias("week"),
        F.col("warehouse").cast("string").alias("warehouse"),
        F.col("forecast").cast("int").alias("forecast"),
        F.col("capacity").cast("int").alias("capacity"),
        F.col("prev_forecast").cast("int").alias("prev_forecast"),
        F.col("change_pct").cast("double").alias("change_pct"),
        F.col("utilization").cast("double").alias("utilization"),
        F.col("status").cast("string").alias("status"),
    )
)

(
    spark_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("llmops_forecasting.inbound_planning.forecast_data")
)

In [0]:
%sql
SELECT warehouse, week, utilization, status
FROM llmops_forecasting.inbound_planning.forecast_data
LIMIT 10

warehouse,week,utilization,status
Aachen,23,0.6461855188734308,normal
Aachen,24,0.686447422877336,normal
Aachen,25,0.49084715545509533,normal
Aachen,26,0.7994014835567158,normal
Aachen,27,0.6036738722103722,normal
Aachen,28,0.67924187917184,normal
Aachen,29,0.644763511720947,normal
Aachen,30,0.6904056965182048,normal
Aachen,31,0.5218872369552068,normal
Aachen,32,0.560843865737056,normal


In [0]:
import pandas as pd
import uuid
from pyspark.sql import functions as F

kb_pd = pd.DataFrame(knowledge_base)

if "id" not in kb_pd.columns:
    kb_pd["id"] = [str(uuid.uuid4()) for _ in range(len(kb_pd))]

kb_pd["week"] = pd.to_numeric(kb_pd["week"], errors="coerce").astype("Int64")
kb_pd["warehouse"] = kb_pd["warehouse"].astype("string")
kb_pd["doc_type"] = kb_pd["doc_type"].astype("string")
kb_pd["text"] = kb_pd["text"].astype("string")
kb_pd["id"] = kb_pd["id"].astype("string")

spark_kb_df = (
    spark.createDataFrame(kb_pd)
    .select(
        "id",
        F.col("week").cast("int").alias("week"),
        "warehouse",
        "doc_type",
        "text",
    )
)
spark_kb_df.write.mode("overwrite").saveAsTable(
    "llmops_forecasting.inbound_planning.knowledge_base"
)

In [0]:
%sql
SELECT doc_type, week, text
FROM llmops_forecasting.inbound_planning.knowledge_base
-- LIMIT 5

doc_type,week,text
